In [31]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [32]:
class BatsmanState(TypedDict):
    runs: int
    balls_played: int
    fours: int
    sixes: int
    calculated_strike_rate: float
    calculated_balls_per_boundary: float
    calculated_boundary_percent: float
    summary: str

In [33]:
def calculate_strike_rate(state: BatsmanState):
    calculated_strike_rate = (state['runs']/ state['balls_played'])*100
    state['calculated_strike_rate'] = calculated_strike_rate
    return {'calculated_strike_rate': calculated_strike_rate}

def calculate_balls_per_boundary(state: BatsmanState):
    calculated_balls_per_boundary = state['balls_played']/ (state['fours'] + state['sixes'])
    state['calculated_balls_per_boundary'] = calculated_balls_per_boundary
    return {'calculated_balls_per_boundary': calculated_balls_per_boundary}

def calculate_boundary_percent(state: BatsmanState):
    calculated_boundary_percent = ((state['fours'] * 4 + state['sixes'] * 6)/state['runs']) * 100
    state['calculated_boundary_percent'] = calculated_boundary_percent
    return {'calculated_boundary_percent': calculated_boundary_percent}

def summary(state: BatsmanState):
    summary_statement = f""" 
    Strike Rate: {state['calculated_strike_rate']}
    Balls per boundary: {state['calculated_balls_per_boundary']}
    Boundary percent: {state['calculated_boundary_percent']}
    """
    state['summary'] = summary_statement
    return {'summary': summary_statement}

In [34]:
graph = StateGraph(BatsmanState)

# ==== node
graph.add_node('calculate_strike_rate', calculate_strike_rate)
graph.add_node('calculate_balls_per_boundary', calculate_balls_per_boundary)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

# ==== edges

graph.add_edge(START, 'calculate_strike_rate')
graph.add_edge(START, 'calculate_balls_per_boundary')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_strike_rate', 'summary')
graph.add_edge('calculate_balls_per_boundary', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)


In [35]:
# === Execute workflow

workflow = graph.compile()

initial_state = {
    'runs': 100,
    'balls_played': 50,
    'fours': 6,
    'sixes': 4
}

workflow.invoke(initial_state)

{'runs': 100,
 'balls_played': 50,
 'fours': 6,
 'sixes': 4,
 'calculated_strike_rate': 200.0,
 'calculated_balls_per_boundary': 5.0,
 'calculated_boundary_percent': 48.0,
 'summary': ' \n    Strike Rate: 200.0\n    Balls per boundary: 5.0\n    Boundary percent: 48.0\n    '}